# PySpark ETL
This notebook focuses on performing ETL (Extract, Transform, Load) operations on the data ingested from the FPL API. We will use PySpark to clean, transform, and prepare the data for model training.

## 1. Include required libraries

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import * # Import all functions
from pyspark.sql.window import Window
import os
from datetime import datetime

## 2. Initialize Spark Session
We initialize a Spark session, which is the entry point to any Spark functionality.

In [4]:
spark = SparkSession.builder.appName("gameweek-prophet-etl").getOrCreate()

your 131072x1 screen size is bogus. expect trouble
25/03/13 21:41:46 WARN Utils: Your hostname, LAPTOP-3B4JAVBH resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/03/13 21:41:46 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/13 21:41:47 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 3. Read Data
We read the CSV files generated by the Data Ingestion Notebook into Spark DataFrames.

In [6]:
data_dir = "../data/raw/fpl_api/20250313_212336/"
elements_df = spark.read.csv(f"{data_dir}elements.csv", header=True, inferSchema=True)
teams_df = spark.read.csv(f"{data_dir}teams.csv", header=True, inferSchema=True)
events_df = spark.read.csv(f"{data_dir}events.csv", header=True, inferSchema=True)
# player_history_df = spark.read.csv(f"{data_dir}player_history.csv", header=True, inferSchema=True)
team_fixtures_df = spark.read.csv(f"{data_dir}team_fixtures.csv", header=True, inferSchema=True)

## 4. Data Cleaning
We'll handle data cleaning tasks such as dealing with missing values and standardizing team/player names.

## 5. Feature Engineering
We'll engineer rolling window features to capture recent and longer-term performance trends. For example, we'll calculate rolling averages for player statistics like total points, goals scored, etc.

In [7]:
# Define a window specification for recent performance (e.g., last 4 weeks)
# We define a window to calculate rolling metrics over the last 4 weeks.
recent_window = Window.partitionBy("element").orderBy("round").rowsBetween(Window.currentRow - 3, Window.currentRow)

# Define a window specification for longer-term performance (e.g., last 12 weeks)
# We define a window to calculate rolling metrics over the last 12 weeks.
long_term_window = Window.partitionBy("element").orderBy("round").rowsBetween(Window.currentRow - 11, Window.currentRow)

# Calculate rolling averages for recent performance
# We calculate rolling averages for points and minutes over the recent window.
player_history_df = player_history_df.withColumn(
    "recent_avg_points", avg("total_points").over(recent_window)
).withColumn(
    "recent_avg_minutes", avg("minutes").over(recent_window)
)

# Calculate rolling averages for longer-term performance
# We calculate rolling averages for points and minutes over the long term window.
player_history_df = player_history_df.withColumn(
    "long_term_avg_points", avg("total_points").over(long_term_window)
).withColumn(
    "long_term_avg_minutes", avg("minutes").over(long_term_window)
)

NameError: name 'player_history_df' is not defined

## 6. Data Transformation
We'll perform any necessary data transformations, such as data type conversions or aggregations.

## 7. Write Data
Finally, we'll write the transformed data to CSV files for use in the subsequent model training notebook.

In [ ]:
# Define output directory
# We define the directory where the processed data will be saved.
data_source = "features"
current_datetime = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"../data/staging/{data_source}/{current_datetime}"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Create the directory if it doesn't exist
# We create the directory if it doesn't exist.
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Write the transformed DataFrames to Parquet files
# We save the processed data as parquet files.
elements_df.write.parquet(f"{output_dir}/elements.parquet", mode="overwrite")
teams_df.write.parquet(f"{output_dir}/teams.parquet", mode="overwrite")
events_df.write.parquet(f"{output_dir}/events.parquet", mode="overwrite")
player_history_df.write.parquet(f"{output_dir}/player_history.parquet", mode="overwrite")
team_fixtures_df.write.parquet(f"{output_dir}/team_fixtures.parquet", mode="overwrite")

print("ETL process complete. Transformed data saved to fpl_processed_data directory.")